# Raw Data Vault

## Setup

### Imports

In [1]:
from pathlib import Path

import duckdb

### Configuration

In [2]:
# Locate the project root (folder containing requirements.txt), independent of cwd
PROJECT_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists()
)

data_vault_path = str(PROJECT_ROOT / "data" / "warehouse" / "data_vault.duckdb")

### DuckDB Connection

In [3]:
con = duckdb.connect(data_vault_path)

In [4]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS raw_vault;
""")

## Hubs

### Customer Hub

In [5]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.hub_customer (
    customer_hk VARCHAR PRIMARY KEY,
    customer_id VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

con.execute("""
INSERT INTO raw_vault.hub_customer
SELECT DISTINCT
    p.customer_hk,
    p.customer_id,
    p.load_date,
    p.record_source
FROM prep.customers p
WHERE p.customer_id IS NOT NULL
  AND TRIM(p.customer_id) <> ''
  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.hub_customer h
      WHERE h.customer_hk = p.customer_hk
  );
""")

### Product Hub

In [6]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.hub_product (
    product_hk VARCHAR PRIMARY KEY,
    product_id VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

con.execute("""
INSERT INTO raw_vault.hub_product
SELECT DISTINCT
    p.product_hk,
    p.product_id,
    p.load_date,
    p.record_source
FROM prep.products p
WHERE p.product_id IS NOT NULL
  AND TRIM(p.product_id) <> ''
  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.hub_product h
      WHERE h.product_hk = p.product_hk
  );
""")

### Hub Seller

In [7]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.hub_seller (
    seller_hk VARCHAR PRIMARY KEY,
    seller_id VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

con.execute("""
INSERT INTO raw_vault.hub_seller
SELECT DISTINCT
    p.seller_hk,
    p.seller_id,
    p.load_date,
    p.record_source
FROM prep.sellers p
WHERE p.seller_id IS NOT NULL
  AND TRIM(p.seller_id) <> ''
  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.hub_seller h
      WHERE h.seller_hk = p.seller_hk
  );
""")

### Hub Order

In [8]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.hub_order (
    order_hk VARCHAR PRIMARY KEY,
    order_id VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

con.execute("""
INSERT INTO raw_vault.hub_order
SELECT DISTINCT
    p.order_hk,
    p.order_id,
    p.load_date,
    p.record_source
FROM prep.orders p
WHERE p.order_id IS NOT NULL
  AND TRIM(p.order_id) <> ''
  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.hub_order h
      WHERE h.order_hk = p.order_hk
  );
""")

### Hub Review

In [9]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.hub_review (
    review_hk VARCHAR PRIMARY KEY,
    review_id VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

con.execute("""
INSERT INTO raw_vault.hub_review
SELECT DISTINCT
    p.review_hk,
    p.review_id,
    p.load_date,
    p.record_source
FROM prep.order_reviews p
WHERE p.review_id IS NOT NULL
  AND TRIM(p.review_id) <> ''
  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.hub_review h
      WHERE h.review_hk = p.review_hk
  );
""")

## Satellites

### Customer Satellite

In [10]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.sat_customer_details (
    customer_hk VARCHAR,
    customer_unique_id VARCHAR,
    customer_zip_code_prefix VARCHAR,
    customer_city VARCHAR,
    customer_state VARCHAR,
    hash_diff VARCHAR,
    load_date TIMESTAMP,
    record_source VARCHAR
);
""")

In [11]:
con.execute("""
INSERT INTO raw_vault.sat_customer_details
WITH latest_sat AS (
    SELECT
        customer_hk,
        hash_diff
    FROM raw_vault.sat_customer_details
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY customer_hk
        ORDER BY load_date DESC
    ) = 1
)
SELECT
    p.customer_hk,
    p.customer_unique_id,
    p.customer_zip_code_prefix,
    p.customer_city,
    p.customer_state,
    p.hash_diff,
    p.load_date,
    p.record_source
FROM prep.customers p
LEFT JOIN latest_sat s
    ON p.customer_hk = s.customer_hk
WHERE s.customer_hk IS NULL
   OR p.hash_diff IS DISTINCT FROM s.hash_diff;
""")

### Product Satellite

In [12]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.sat_product_details (
    product_hk VARCHAR NOT NULL,
    product_category_name VARCHAR,

    product_name_length INTEGER,
    product_description_length INTEGER,
    product_photos_qty INTEGER,

    product_weight_g INTEGER,
    product_length_cm INTEGER,
    product_height_cm INTEGER,
    product_width_cm INTEGER,

    hash_diff VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [13]:
con.execute("""
INSERT INTO raw_vault.sat_product_details
WITH latest_sat AS (
    SELECT
        product_hk,
        hash_diff
    FROM raw_vault.sat_product_details
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY product_hk
        ORDER BY load_date DESC
    ) = 1
)
SELECT
    p.product_hk,
    p.product_category_name,
    p.product_name_length,
    p.product_description_length,
    p.product_photos_qty,
    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,
    p.hash_diff,
    p.load_date,
    p.record_source
FROM prep.products p
LEFT JOIN latest_sat s
    ON p.product_hk = s.product_hk
WHERE s.product_hk IS NULL
   OR p.hash_diff IS DISTINCT FROM s.hash_diff;
""")

### Seller Satellite

In [14]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.sat_seller_details (
    seller_hk VARCHAR NOT NULL,

    seller_zip_code_prefix VARCHAR,
    seller_city VARCHAR,
    seller_state VARCHAR,

    hash_diff VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [15]:
con.execute("""
INSERT INTO raw_vault.sat_seller_details
WITH latest_sat AS (
    SELECT
        seller_hk,
        hash_diff
    FROM raw_vault.sat_seller_details
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY seller_hk
        ORDER BY load_date DESC
    ) = 1
)
SELECT
    p.seller_hk,
    p.seller_zip_code_prefix,
    p.seller_city,
    p.seller_state,
    p.hash_diff,
    p.load_date,
    p.record_source
FROM prep.sellers p
LEFT JOIN latest_sat s
    ON p.seller_hk = s.seller_hk
WHERE s.seller_hk IS NULL
   OR p.hash_diff IS DISTINCT FROM s.hash_diff;
""")

### Order Satellite

In [16]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.sat_order_details (
    order_hk VARCHAR NOT NULL,

    order_status VARCHAR,

    order_purchase_timestamp TIMESTAMP,
    order_approved_at TIMESTAMP,
    order_delivered_carrier_date TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,

    hash_diff VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [17]:
con.execute("""
INSERT INTO raw_vault.sat_order_details
WITH latest_sat AS (
    SELECT
        order_hk,
        hash_diff
    FROM raw_vault.sat_order_details
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_hk
        ORDER BY load_date DESC
    ) = 1
)
SELECT
    p.order_hk,
    p.order_status,
    p.order_purchase_timestamp,
    p.order_approved_at,
    p.order_delivered_carrier_date,
    p.order_delivered_customer_date,
    p.order_estimated_delivery_date,
    p.hash_diff,
    p.load_date,
    p.record_source
FROM prep.orders p
LEFT JOIN latest_sat s
    ON p.order_hk = s.order_hk
WHERE s.order_hk IS NULL
   OR p.hash_diff IS DISTINCT FROM s.hash_diff;
""")

### Review Satellite

In [18]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.sat_review_details (
    review_hk VARCHAR NOT NULL,

    review_score INTEGER,
    review_comment_title VARCHAR,
    review_comment_message VARCHAR,
    review_creation_date TIMESTAMP,
    review_answer_timestamp TIMESTAMP,

    hash_diff VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [19]:
con.execute("""
INSERT INTO raw_vault.sat_review_details
WITH latest_sat AS (
    SELECT
        review_hk,
        hash_diff
    FROM raw_vault.sat_review_details
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY review_hk
        ORDER BY load_date DESC
    ) = 1
)
SELECT
    p.review_hk,
    p.review_score,
    p.review_comment_title,
    p.review_comment_message,
    p.review_creation_date,
    p.review_answer_timestamp,
    p.hash_diff,
    p.load_date,
    p.record_source
FROM prep.order_reviews p
LEFT JOIN latest_sat s
    ON p.review_hk = s.review_hk
WHERE s.review_hk IS NULL
   OR p.hash_diff IS DISTINCT FROM s.hash_diff;
""")

### Order Item Link Satellite

In [20]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.sat_order_item_details (
    order_item_hk VARCHAR NOT NULL,

    shipping_limit_date TIMESTAMP,
    price DECIMAL(18,2),
    freight_value DECIMAL(18,2),

    hash_diff VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [21]:
con.execute("""
INSERT INTO raw_vault.sat_order_item_details
WITH latest_sat AS (
    SELECT
        order_item_hk,
        hash_diff
    FROM raw_vault.sat_order_item_details
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_item_hk
        ORDER BY load_date DESC
    ) = 1
)
SELECT
    p.order_item_hk,
    p.shipping_limit_date,
    p.price,
    p.freight_value,
    p.hash_diff,
    p.load_date,
    p.record_source
FROM prep.order_items p
LEFT JOIN latest_sat s
    ON p.order_item_hk = s.order_item_hk
WHERE s.order_item_hk IS NULL
   OR p.hash_diff IS DISTINCT FROM s.hash_diff;
""")

### Order Payment Link Satellite

In [22]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.sat_order_payment_details (
    order_payment_hk VARCHAR NOT NULL,

    payment_type VARCHAR,
    payment_installments INTEGER,
    payment_value DECIMAL(18,2),

    hash_diff VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [23]:
con.execute("""
INSERT INTO raw_vault.sat_order_payment_details
WITH latest_sat AS (
    SELECT
        order_payment_hk,
        hash_diff
    FROM raw_vault.sat_order_payment_details
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_payment_hk
        ORDER BY load_date DESC
    ) = 1
)
SELECT
    p.order_payment_hk,
    p.payment_type,
    p.payment_installments,
    p.payment_value,
    p.hash_diff,
    p.load_date,
    p.record_source
FROM prep.order_payments p
LEFT JOIN latest_sat s
    ON p.order_payment_hk = s.order_payment_hk
WHERE s.order_payment_hk IS NULL
   OR p.hash_diff IS DISTINCT FROM s.hash_diff;
""")

## Links

### Link Order Customer

In [24]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.link_order_customer (
    order_customer_hk VARCHAR PRIMARY KEY,
    order_hk VARCHAR NOT NULL,
    customer_hk VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [25]:
con.execute("""
INSERT INTO raw_vault.link_order_customer
SELECT DISTINCT
    p.order_customer_hk,
    p.order_hk,
    p.customer_hk,
    p.load_date,
    p.record_source

FROM prep.orders p

WHERE p.order_id IS NOT NULL
  AND p.customer_id IS NOT NULL

  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.link_order_customer l
      WHERE l.order_customer_hk = p.order_customer_hk
  );
""")

### Link Order Review

In [26]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.link_order_review (
    order_review_hk VARCHAR PRIMARY KEY,
    order_hk VARCHAR NOT NULL,
    review_hk VARCHAR NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [27]:
con.execute("""
INSERT INTO raw_vault.link_order_review
SELECT DISTINCT
    p.order_review_hk,
    p.order_hk,
    p.review_hk,
    p.load_date,
    p.record_source

FROM prep.order_reviews p

WHERE p.order_id IS NOT NULL
  AND p.review_id IS NOT NULL

  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.link_order_review l
      WHERE l.order_review_hk = p.order_review_hk
  );
""")

### Link Order Item

In [28]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.link_order_item (
    order_item_hk VARCHAR PRIMARY KEY,
    order_hk VARCHAR NOT NULL,
    product_hk VARCHAR NOT NULL,
    seller_hk VARCHAR NOT NULL,
    order_item_id INTEGER NOT NULL,
    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [29]:
con.execute("""
INSERT INTO raw_vault.link_order_item
SELECT DISTINCT
    p.order_item_hk,
    p.order_hk,
    p.product_hk,
    p.seller_hk,
    p.order_item_id,
    p.load_date,
    p.record_source

FROM prep.order_items p

WHERE p.order_id IS NOT NULL
  AND p.product_id IS NOT NULL
  AND p.seller_id IS NOT NULL

  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.link_order_item l
      WHERE l.order_item_hk = p.order_item_hk
  );
""")

### Link Order Payments

In [30]:
con.execute("""
CREATE TABLE IF NOT EXISTS raw_vault.link_order_payment (
    order_payment_hk VARCHAR PRIMARY KEY,
    order_hk VARCHAR NOT NULL,
    payment_sequential INTEGER NOT NULL,

    load_date TIMESTAMP NOT NULL,
    record_source VARCHAR NOT NULL
);
""")

In [31]:
con.execute("""
INSERT INTO raw_vault.link_order_payment
SELECT DISTINCT
    p.order_payment_hk,
    p.order_hk,
    p.payment_sequential,
    p.load_date,
    p.record_source

FROM prep.order_payments p

WHERE p.order_id IS NOT NULL
  AND p.payment_sequential IS NOT NULL

  AND NOT EXISTS (
      SELECT 1
      FROM raw_vault.link_order_payment l
      WHERE l.order_payment_hk = p.order_payment_hk
  );
""")

## List Tables

In [32]:
con.execute("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'raw_vault'
ORDER BY table_name;
""").df()

,table_name
0,hub_customer
1,hub_order
2,hub_product
3,hub_review
4,hub_seller
5,link_order_customer
6,link_order_item
7,link_order_payment
8,link_order_review
9,sat_customer_details
